# Task 2 — ML Analysis

Time-series gaps, message classification, topic modelling, sentiment analysis, and MLflow logging.

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import nltk
import seaborn as sns

rpath = os.path.abspath('..')
if rpath not in sys.path:
    sys.path.insert(0, rpath)

from src.loader import SlackDataLoader
from src.analysis.time_gaps import get_time_gap_histograms
from src.models.classifier import train_message_classifier, build_labeled_frame
from src.models.topic_model import get_top_topics_by_channel
from src.models.sentiment import daily_sentiment_trend

nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('vader_lexicon', quiet=True)

sns.set_theme(style='whitegrid')
DATA_PATH = os.path.join(rpath, 'data', 'anonymized')
MLFLOW_DIR = os.path.join(rpath, 'mlruns')
os.makedirs(MLFLOW_DIR, exist_ok=True)

In [ ]:
loader = SlackDataLoader(DATA_PATH)
df = loader.get_all_messages()
print(f'Loaded {len(df):,} messages from {df["channel"].nunique()} channels')

## 1. Time difference histograms

In [ ]:
gaps = get_time_gap_histograms(df)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (title, series) in zip(axes.flatten(), gaps.items()):
    if series.empty:
        ax.set_title(f'{title} (no data)')
        continue
    clipped = series[series <= series.quantile(0.99)]
    ax.hist(clipped, bins=40, color='steelblue', edgecolor='white')
    ax.set_title(title.replace('_', ' ').title())
    ax.set_xlabel('Seconds between consecutive events')
    ax.set_ylabel('Frequency')
plt.tight_layout()
plt.show()

## 2. Message classification

In [ ]:
pipeline, labeled_df, report = train_message_classifier(df)
print('Label distribution:')
print(labeled_df['label'].value_counts())
print(f"\nTest accuracy: {report['accuracy']:.3f}")

## 3. Top 10 topics per channel

In [ ]:
topics_df = get_top_topics_by_channel(df, num_topics=10)
for channel in topics_df['channel'].unique()[:5]:
    print(f'\n### #{channel}')
    display(topics_df[topics_df['channel'] == channel].head(10))

## 4. Sentiment trend over days since training start

In [ ]:
sentiment_df = daily_sentiment_trend(df)

plt.figure(figsize=(12, 5))
plt.plot(sentiment_df['days_since_start'], sentiment_df['sentiment'], marker='o', linewidth=2)
plt.title('Average Daily Sentiment Over Time')
plt.xlabel('Days since training start')
plt.ylabel('Compound sentiment score')
plt.tight_layout()
plt.show()

sentiment_df.head()

## Observations

- Describe the shape of each time-gap histogram.
- Which message types dominate the classifier output?
- What themes appear in the busiest channels?
- Does sentiment improve, decline, or fluctuate over the training period?